# **Pyspark vs SQL**

Regla mental que debes repetir

- SQL = Declarativo
- PySpark = Transformaciones encadenadas Sobre DataFrames


In [0]:
%sql
SELECT *
FROM samples.bakehouse.sales_customers
LIMIT 5;

In [0]:
df = spark.read.table("samples.bakehouse.sales_customers")

df_selected = df.select("*")

df_limited = df_selected.limit(5)

df_limited.display()





#Distinct

In [0]:
%sql
select distinct country
from samples.bakehouse.sales_customers


In [0]:
df_distinct = df.select("country").distinct()

display(df_distinct)

# WHERE / FILTER


In [0]:
%sql
SELECT *
FROM samples.bakehouse.sales_customers
WHERE 1=1 
AND country = 'Japan' 
AND gender = 'female'
LIMIT 5;



In [0]:
from pyspark.sql.functions import col

df_filtered = df.filter(
    (col("country") == "Japon") & 
    (col("gender") == "female" )
    )

display(df_filtered)

# ORDER BY + LIMIT

In [0]:
%sql
SELECT *
FROM samples.bakehouse.sales_customers
ORDER BY first_name ASC
LIMIT 5;


In [0]:
df_sorted = df.orderBy(col("first_name"))

display(df_sorted.limit(5))

# CREATE COLUMN

In [0]:
%sql
SELECT 
concat(first_name, ' ', last_name) AS full_name,
concat_ws(' , ' , city, state, country, continent) AS address, 
35 AS age,
"ramdom comment" AS comment
FROM samples.bakehouse.sales_customers
LIMIT 5;

In [0]:
from pyspark.sql.functions import col, concat, lit

df_plus_columns = df.select(
    concat(col("first_name"), lit(" "), col("last_name")).alias("full_name"),
    lit(True).alias("is_plus"),
    lit("random comment").alias("comment")).limit(5)

display(df_plus_columns)






# GROUP BY + AGGREGATION

In [0]:
%sql
SELECT country, count(*) as cantidad
FROM samples.bakehouse.sales_franchises
GROUP BY country
ORDER BY cantidad DESC
LIMIT 5

In [0]:
df2 = spark.read.table("samples.bakehouse.sales_franchises")

from pyspark.sql.functions import count

df_grouped = (df2.groupBy("country")
              .agg(count("*").alias("count"))
              .orderBy(col("count").desc())  
               )
                

display(df_grouped.limit(5))

# JOIN (CLAVE PARA SILVER)

In [0]:
%sql
SELECT
 concat(c.first_name, ' ', c.last_name) AS full_name,
 c.country,
 c.gender,
 s.product,
 s.quantity,
 s.paymentMethod,
 s.dateTime
FROM samples.bakehouse.sales_transactions s  
JOIN samples.bakehouse.sales_customers c
ON s.customerID = c.customerID
limit 5


In [0]:
df_sales = spark.read.table("samples.bakehouse.sales_transactions")

df_joined = ( df
             .join(df_sales, on="customerID", how="inner")
             .select("*")
             )



display(df_joined.limit(5)) 


# convertir un valor de un tipo de dato a otro

In [0]:
%sql
SELECT
  cast(transactionID AS int) AS id,
  cast(customerID AS int) AS customer,
  cast(franchiseID AS int) AS store,
  cast(dateTime AS date) AS date,
  cast(product AS string) AS product
FROM samples.bakehouse.sales_transactions
LIMIT 5;

In [0]:

df_casted = df_sales.select(
    col("transactionID").cast("int").alias("transactionID"),
    col("customerID").cast("int").alias("customerID"),
    col("franchiseID").cast("int").alias("franchiseID"),
    col("dateTime").cast("timestamp").alias("dateTime"),
    col("product").cast("string").alias("product")
)

display(df_casted.limit(5)) 


In [0]:
df_test = df_casted.withColumn("test", lit("test"))

display(df_test.limit(5))   




In [0]:
df_new = df_test.drop("test")

display(df_new.limit(5))